In [1]:
# Get raw data
with open('input/18.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
import numpy as np

In [3]:
# Part 1
def reach_iter(reach, layout):
    return np.where(((layout == '.')
                     & (reach == -1)
                     & ((z:=np.max(np.stack(((y:=np.pad(reach, 1, constant_values=-1))[:-2, 1:-1], 
                                             y[2:, 1:-1], 
                                             y[1:-1, :-2], 
                                             y[1:-1, 2:])), 
                                   axis=0)) >= 0)),
                    z+1, 
                    reach)
    
def get_reach(layout, startpos):
    reach = -np.ones_like(layout, dtype=int)
    reach[*startpos] = 0
    while np.any(reach != (reach:=reach_iter(reach, layout))):  pass
    return reach

layout = np.array([[['.',c][c in '.#'] for c in r] for r in rawinput.split('\n')])
doors = np.array([[['',c][64<ord(c)<91] for c in r] for r in rawinput.split('\n')])
pts = {c:(i,j) 
       for i,r in enumerate(rawinput.split('\n')) 
       for j,c in enumerate(r) 
       if 96<ord(c)<123}
pts['START'] = ((z:=rawinput.replace('\n','').index('@')) // (w:=layout.shape[1]), z%w)

reach = {k: get_reach(layout, pts[k]) for k in pts}
paths = dict()
for i,j in [[i,j] for i in pts for j in pts if j>i]:
    for k in {i,j}-{*paths}:
        paths[k] = dict()
    paths[i][j] = ((dist:=reach[i][*pts[j]].item()), 
                   {k.lower()
                    for filt in [reach[i]+reach[j] == dist]
                    for z in [doors[filt][np.argsort(reach[i][filt])]]
                    for k in z[z!='']})
    if i != 'START':
        paths[j][i] = paths[i][j]
    
known = dict()
def get_min_steps(pos='START', collected=None):
    if collected is None:
        collected = set()
    dkey = f"{pos}|{''.join(sorted(collected-{pos}))}"
    if dkey not in known:
        if {*pts}-{'START'}-collected:
            known[dkey] = min([v[0]+get_min_steps(k,collected|{k}) 
                               for k,v in paths[pos].items() 
                               if k not in collected
                               and v[1].issubset(collected)])
        else:
            known[dkey] = 0
    return known[dkey]

get_min_steps()

5102

In [4]:
# Part 2
layout[(c:=pts['START'])[0]-1:c[0]+2, c[1]-1:c[1]+2] = np.array([*'.#.###.#.']).reshape(3,3)
pts.update({f'START{i}': tuple([sum(k) 
                                for k in zip(pts['START'], 
                                             [int(j)*2-1 
                                              for j in bin(i+4)[-2:]])]) 
            for i in range(4)})
del(pts['START'])

reach = {k: get_reach(layout, pts[k]) for k in pts}

paths = dict()
for i,j in [[i,j] for i in pts for j in pts if j>i and not j.startswith('START')]:
    for k in {i,j}-{*paths}:
        paths[k] = dict()
    paths[i][j] = ((dist:=reach[i][*pts[j]].item()), 
                   {k.lower()
                    for filt in [reach[i]+reach[j] == dist]
                    for z in [doors[filt][np.argsort(reach[i][filt])]]
                    for k in z[z!='']})
    if not i.startswith('START'):
        paths[j][i] = paths[i][j]

known = dict()
def get_min_steps(pos=None, collected=None):
    if pos is None:
        pos = [f'START{i}' for i in range(4)]
    if collected is None:
        collected = set()
    dkey = f"{'|'.join(pos)}|{''.join(sorted(collected-{*pos}))}"
    if dkey not in known:
        if {*pts}-collected-{f'START{i}' for i in range(4)}:
            known[dkey] = min([v[0]+get_min_steps([[v,k][u==i] for u,v in enumerate(pos)],
                                                  collected|{k})
                               for i in range(4)
                               for k,v in paths[pos[i]].items()
                               if v[0] >= 0
                               and k not in collected
                               and v[1].issubset(collected)])
        else:
            known[dkey] = 0
    return known[dkey]

get_min_steps()

2282